# Lecture 9: policy experiments, welfare, and the transition

**ECU44291 Quantitative Macroeconomics, Week 10.**

Policy experiments are kind of why we got in the game. The value of quantitative macro is to be able to provide some kind of answer of whether one scenario is 'better' than another.  Today we see two ways to pay for a pension, one way of assessing which situation makes people "better off" (the consumption equivalent), and what happens to the people alive when the reform arrives. Most everything here is the two-period OLG economy from Lectures 2 and 7, where each step is a line or two. The sixty-cohort results are on the slides; building that economy is milestone 3.

This is the complete notebook, and I walk through most of it in lecture.

## The two-period OLG economy, with two kinds of pension

As in Lecture 7: log utility, $\beta = 0.9$, $\alpha = 0.3$, $\delta = 0$, $\tau_L = 0.2$, $\tau_K = 0.15$, one young and one old household, $L = 1$. A period is a generation, about 30 years.

Two pension schemes, each paid for by a contribution on the wage of the young:

- **pay-as-you-go**, rate $\tau_p$: this period's contributions are paid out to this period's old, so the old receive $b = \tau_p w$;
- **funded**, rate $\tau_f$: the contribution $\tau_f w$ goes into an account that is invested in capital, and is paid back with its return, $R\,\tau_f w$, when the household is old.

In [ ]:
import numpy as np
from scipy.optimize import brentq

def params(**overrides):
    p = dict(alpha=0.3, beta=0.9, delta=0.0, tau_L=0.2, tau_K=0.15, tau_p=0.0, tau_f=0.0)
    p.update(overrides)
    return p

def prices(K, p):
    """Wage and the household's after-tax gross return at capital K (L = 1)."""
    w = (1 - p["alpha"]) * K ** p["alpha"]
    R = 1 + (1 - p["tau_K"]) * (p["alpha"] * K ** (p["alpha"] - 1) - p["delta"])
    return w, R

def saving(w, R, p):
    """Private saving of the young (log utility). The funded account is saved for them on top of this."""
    income_y = (1 - p["tau_L"] - p["tau_p"] - p["tau_f"]) * w
    income_o = p["tau_p"] * w + R * p["tau_f"] * w        # pension when old: pay-as-you-go, plus the account
    return (p["beta"] * R * income_y - income_o) / ((1 + p["beta"]) * R)

def steady_state(p):
    """Capital is private saving plus the funded account: K = s + tau_f w."""
    def gap(K):
        w, R = prices(K, p)
        return saving(w, R, p) + p["tau_f"] * w - K
    K = brentq(gap, 1e-3, 2.0, xtol=1e-12)
    w, R = prices(K, p)
    s = saving(w, R, p)
    account = p["tau_f"] * w
    c_y = (1 - p["tau_L"] - p["tau_p"] - p["tau_f"]) * w - s
    c_o = R * (s + account) + p["tau_p"] * w
    return dict(K=K, w=w, R=R, s=s, account=account, c_y=c_y, c_o=c_o)

## Task 1: pay-as-you-go against funded, in steady state

Both schemes take 30% of the young's wage.

**Question:** compared with an economy with no pension at all, will the funded economy have more capital, less, or the same?

In [ ]:
# your code here


*Answer:* The same. The household wanted to save $\frac{\beta}{1+\beta}(1-\tau_L)\,w$ in total. The account saves $\tau_f w$ of that for it, at the same return, so it saves exactly that much less on its own: private saving falls one for one. Capital, prices and consumption are all as with no pension. (This needs the account to be no bigger than what the household would save anyway: here 0.3 against 0.379 of the wage.)

Pay-as-you-go is different: the contribution is paid straight to today's old, so it is never invested. Capital falls by almost two thirds.

## Task 2: One number that defines what we mean by "better off"

Lifetime utility of a household born into each steady state is $U = \ln c_y + \beta \ln c_o$. Utility has no units, so we convert the difference into consumption: the **consumption equivalent** $\lambda$ is the percentage by which you would have to raise the pay-as-you-go household's consumption, in both periods, to make it as well off as the funded household. With log utility,
$$\ln\big((1+\lambda) c_y\big) + \beta \ln\big((1+\lambda) c_o\big) = U + (1+\beta)\ln(1+\lambda), \qquad\text{so}\qquad \lambda = \exp\!\Big(\frac{U_{\text{new}} - U_{\text{base}}}{1+\beta}\Big) - 1 .$$

**Question:** is $\lambda$ for funded against pay-as-you-go above or below 10%?

In [ ]:
# your code here


A newborn in the funded steady state is as well off as a pay-as-you-go newborn whose consumption was 49% higher in both periods. Large, because a period is 30 years and the pay-as-you-go pension cut capital by almost two thirds (these results are a bit wonky in two-period world).

## Task 2, continued: the same number from $V$, on a sixty-age household

With sixty ages there is no closed form for lifetime utility, but your solver already computes it: the lifetime utility of a newborn with no assets is $V_1(0)$, the first row of `V` at the first grid point, `V[0, 0]`. With $\sigma = 2$, $\lambda = V^{\text{base}}_1(0) / V^{\text{reform}}_1(0) - 1$.

Here: Lecture 3's household on the grid, at fixed prices, $r = 5.45\%$ and $w = 1.058$ (the pay-as-you-go steady state on the slides; finding them is M3). Pay-as-you-go with repl $= 0.4$, so with equal cohorts the budget needs $\tau = 0.4 \times 15/45$. Funded: the same $\tau$ into an account that earns $r$ and is paid back as an annuity $P$ over the 15 years of retirement.

**Before you run:** at these prices, is the funded household better off, and by more or less than the 49% in two periods?

In [ ]:
# your code here


About 7%: the same contributions earn $r$ instead of nothing, so the pension each year is 2.38 instead of 0.42, and the household saves less of its own. Smaller than in two periods, because this is only the return, at prices that do not move. The slides show what happens when they do.

Both numbers compare households born long after any reform. Nobody alive when the reform happens is in them.

## Task 3: the reform as a path

The economy has been pay-as-you-go for a long time. At date 0 the government announces, as a surprise, that from now on pensions are funded. By Task 1, a funded pension is the same as saving for yourself, so from date 0 each young household simply saves for its own old age.

The old at date 0 paid into pay-as-you-go when they were young, and expect $b = \tau_p w_0$. Someone has to pay it. Here the young at date 0 pay them one last time (`honour=True`), and also save for themselves.

Capital at date 0 is what the old saved when young, so it is the pay-as-you-go steady state's. From there the path runs forward one date at a time: prices from $K_t$; the young save by the rule; their saving is $K_{t+1}$. With log utility and no pension when old, the rule needs only today's wage, so no future price has to be guessed.

**Question:** is the household young at the reform better or worse off than if pay-as-you-go had stayed?

In [ ]:
# your code here


The young at the reform pay twice: the old's pension, and their own saving. They are 21% worse off. Every later cohort gains, more and more as capital builds up, towards the steady state's 49%. The old at the reform are untouched.

A steady-state comparison reports only the 49%.

## Task 4: who pays for the last pension

**The edit:** in the cell above, change `honour=True` to `honour=False`. The pension stops at date 0, and the old at the reform get only their own savings.

**Question:** who loses now, and who gains compared with Task 3?

With `honour=False`, the old at the reform lose 19% and the young at the reform gain 15%. The cost of ending pay-as-you-go has not gone away: it has moved from one generation to the other. Spreading it over later cohorts with government debt would move it again; it cannot make it disappear.

## Task 5 (homework): the other direction

Start from the steady state with **no** pension. At date 0, as a surprise, a pay-as-you-go pension with $\tau_p = 0.3$ starts: from date 0 the young pay $\tau_p w_t$ and receive $\tau_p w_{t+1}$ when old, and the old at date 0 receive $\tau_p w_0$ without having paid anything.

Compute the path of $K$ and the consumption equivalent of each birth cohort against staying with no pension.

One difference from Task 3: the young now get a pension when old, $\tau_p w_{t+1}$, and the rule's second term needs $w_{t+1}$ and $R_{t+1}$, which depend on $K_{t+1}$, which is their own saving. So at each date, find the saving $s_t$ that is consistent with the prices it creates, with `brentq`.

In [ ]:
# your code here


## Before the next lecture

- **M3, the engine and its calibration: Friday 27 November, 17:00**, tag `m3`.
- **Tutorial 4, Wednesday 25 November:** bring one figure from your extension.
- Task 5 above.
- Next lecture, Thursday 26 November: income risk in the life-cycle household.